# Try Vikhr for few-shot simplification

In [2]:
import pathlib as pth

base_location  = pth.Path.cwd().parent.parent

vikhr_location  = base_location / "models" / "vikhr" / "Vikhr-7B-instruct_0.4-Q6_K.gguf"

## Load Model

In [12]:
from langchain_community.llms import LlamaCpp


model = LlamaCpp(
            model_path=vikhr_location.as_posix(),
            n_gpu_layers=16,
            n_batch=512,
            temperature=0.8,
            max_tokens=256,
            top_p=5,
            verbose=False,
            n_ctx=8192,
            f16_kv=True,
            repeat_penalty=1.1,
        )

## Let's try the zero-shot simplification tool by levels

In [13]:
def format_prompt(user_text: str, level: int) -> str:
    if level == 1:
        few_shot_examples = (
            """<|im_start|>user\n"""
            "Упрости следующий текст на русском языке: "
            "Первый будет игровым переложением мультсериала \"Бэтмен по ту сторону\", который дебютировал в прошлом году на WB Kids Network.<|im_end|>\n"
            "<|im_start|>assistant\n"
            "Первый фильм будет игровым переложением мультсериала «Бэтмен по ту сторону». Мультсериал начал выходить в прошлом году.<|im_end|>\n"
            "<|im_start|>user\n"
            "Упрости следующий текст на русском языке: "
            "Фильм будет ставить режиссер Боаз Якин (\"Свежий\").<|im_end|>\n"
            "<|im_start|>assistant\n"
            "Режиссёр Боаз Якин («Свежий») будет ставить фильм.<|im_end|>\n"
            "<|im_start|>user\n"
            "Упрости следующий текст на русском языке: "
            "Он уточнил, что на данный момент уже заключено семь государственных контрактов на выполнение ремонтных работ дорог в районах области, их стоимость оценивается в 3,2 миллиарда рублей.<|im_end|>\n"
            "<|im_start|>assistant\n"
            "Сейчас уже заключили семь контрактов на выполнение работ в области. Это стоит 3,2 миллиарда рублей.<|im_end|>\n"
        )

        prompt = (
            """<|im_start|>system\n"""
            "Ты — помощник для упрощения текста на русском языке. "
            "Тебе дан текст на русском языке, и ты должна предоставить его упрощённую версию на русском языке. "
            "Избегай объединения слишком большого количества деталей в одно предложение; распределяй их по нескольким предложениям, если это необходимо. "
            "Сохраняй важную информацию, такую как имена, национальности и роли. Не удаляй важные детали. "
            "Перефразируй предложения, удаляя причастные и деепричастные обороты. "
            "По возможности заменяй пассивный залог на активный. "
            "Если предложение состоит только из существительного, добавь глагол. "
            "Редкие или малоупотребительные слова заменяй на более распространённые. "
            "Где уместно, удаляй или заменяй иностранные слова. "
            "Неясные фразы заменяй более конкретными, легко понимаемыми словами. "
            "По возможности избегай слов, имеющих паронимы. "
            + few_shot_examples +
            "Используй только русский язык, английский запрещён.<|im_end|>\n"
            "<|im_start|>user\n"
            "Упрости следующий текст на русском языке: {user_text}<|im_end|>\n"
            "<|im_start|>assistant\nУпрощенный текст: "
        )

    elif level == 2:
        few_shot_examples = (
            """<|im_start|>user\n"""
            "Упрости следующий текст на русском языке: "
            "Первый будет игровым переложением мультсериала \"Бэтмен по ту сторону\", который дебютировал в прошлом году на WB Kids Network.<|im_end|>\n"
            "<|im_start|>assistant\n"
            "Первый фильм будет версией мультфильма «Бэтмен по ту сторону». Мультфильм вышел в прошлом году.<|im_end|>\n"
            "<|im_start|>user\n"
            "Упрости следующий текст на русском языке: "
            "Фильм будет ставить режиссер Боаз Якин (\"Свежий\").<|im_end|>\n"
            "<|im_start|>assistant\n"
            "Режиссёр Боаз Якин будет ставить фильм. Он известен по фильму «Свежий».<|im_end|>\n"
            "<|im_start|>user\n"
            "Упрости следующий текст на русском языке: "
            "Он уточнил, что на данный момент уже заключено семь государственных контрактов на выполнение ремонтных работ дорог в районах области, их стоимость оценивается в 3,2 миллиарда рублей.<|im_end|>\n"
            "<|im_start|>assistant\n"
            "Заключили семь контрактов на выполнение работ. Это стоит 3,2 миллиарда рублей.<|im_end|>\n"
        )

        prompt = (
            """<|im_start|>system\n"""
            "Ты — помощник для упрощения текста на русском языке. "
            "Тебе дан текст на русском языке, и ты должна предоставить его упрощённую версию на русском языке. "
            "Упрощай сложные или составные предложения, разбивая их на короткие, с длиной не более семи слов. "
            "Убедись, что каждое предложение содержит только одну идею. "
            "Избегай причастных и деепричастных оборотов, отдавай предпочтение активному залогу вместо пассивного. "
            "Сохраняй важную информацию, такую как имена, национальности и роли. Не удаляй важные детали. "
            "Удаляй ненужные иностранные слова (например, названия брендов) и заменяй редкие или длинные слова на более простые и короткие. "
            "Упрощай двойственные по смыслу фразы, используя конкретные и ясные слова. "
            "Удаляй незначительные детали, не добавляющие важного смысла, но оставляй ключевую информацию. "
            + few_shot_examples +
            "Используй только русский язык, английский запрещён.<|im_end|>\n"
            "<|im_start|>user\n"
            "Упрости следующий текст на русском языке: {user_text}<|im_end|>\n"
            "<|im_start|>assistant\nУпрощенный текст: "
        )

    elif level == 3:
        few_shot_examples = (
            """<|im_start|>user\n"""
            "Упрости следующий текст на русском языке: "
            "Первый будет игровым переложением мультсериала \"Бэтмен по ту сторону\", который дебютировал в прошлом году на WB Kids Network.<|im_end|>\n"
            "<|im_start|>assistant\n"
            "Первый фильм будет снять по мультфильму «Бэтмен по ту сторону». Мультфильм вышел в прошлом году.<|im_end|>\n"
            "<|im_start|>user\n"
            "Упрости следующий текст на русском языке: "
            "Фильм будет ставить режиссер Боаз Якин (\"Свежий\").<|im_end|>\n"
            "<|im_start|>assistant\n"
            "Фильм снимет режиссёр Боаз Якин. Он известен по фильму «Свежий».<|im_end|>\n"
            "<|im_start|>user\n"
            "Упрости следующий текст на русском языке: "
            "Он уточнил, что на данный момент уже заключено семь государственных контрактов на выполнение ремонтных работ дорог в районах области, их стоимость оценивается в 3,2 миллиарда рублей.<|im_end|>\n"
            "<|im_start|>assistant\n"
            "Уже заключили договоры на ремонт. Он стоит 3.2 миллиарда рублей.<|im_end|>\n"
        )
        prompt = (
            """<|im_start|>system\n"""
            "Ты — помощник для упрощения текста на русском языке. "
            "Тебе дан текст на русском языке, и ты должна предоставить его упрощённую версию на русском языке. "
            "Твоя задача — сделать текст максимально простым. "
            "Каждое предложение должно содержать только одну идею и быть длиной не более пяти слов. "
            "Удаляй или заменяй иностранные слова (такие как имена, места или бренды), избегай незначительных деталей. "
            "Исключай числительные и удаляй ненужные подробности. "
            "Используй только именительный и родительный падежи для существительных и только настоящее или прошедшее время для глаголов. "
            "Избегай пассивного залога и инверсии слов. "
            "Редкие или малоупотребительные слова заменяй на более распространённые. "
            "Заменяй сложные фразы на общеупотребительные выражения, клише или идиомы. "
            "Удаляй лишние детали (если это возможно без искажения смысла предложения) и максимально упрощай неясные фразы."
            + few_shot_examples +
            "Используй только русский язык, английский запрещён.<|im_end|>\n"
            "<|im_start|>user\n"
            "Упрости следующий текст на русском языке: {user_text}<|im_end|>\n"
            "<|im_start|>assistant\nУпрощенный текст: "
        )

    return prompt.format(user_text=user_text)


### 1 Level

In [14]:
prompt = format_prompt(user_text="Россиянка Елена Максимова одержала победу в международном конкурсе «Миссис Вселенная».", level=1)

model.invoke(str(prompt))

'1) Российская конкурсантка по имени Елена Максимова стала победительницей международного конкурса "Миссис Вселенная".'

### 2 Level

In [15]:
prompt = format_prompt(user_text="Россиянка Елена Максимова одержала победу в международном конкурсе «Миссис Вселенная».", level=2)

model.invoke(str(prompt))

'1) Российская женщина по имени Елена Максимова выиграла конкурс Миссис Вселенная; 2). Конкурс назывался Международный Мистер и Миссис Вселенной.'

### 3 Level

In [16]:
prompt = format_prompt(user_text="Россиянка Елена Максимова одержала победу в международном конкурсе «Миссис Вселенная».", level=3)

model.invoke(str(prompt))

'20-летняя российская модель Елена Максимова выиграла международный конкурс красоты "Mrs Universe".'

In [3]:
import pandas as pd

data_location = base_location / "data" / "RuSimpleSentAphasia.csv"
data = pd.read_csv(data_location.as_posix())

data.head()

,source,level 1,level 2,level 3
0,Россиянка Елена Максимова одержала победу в ме...,Россиянка Елена Максимова победила в конкурсе ...,Россиянка победила в конкурсе «Миссис Вселенная».,Россиянка победила в конкурсе «Миссис Вселенная».
1,Представительница России впервые завоевала это...,"В прессе сказали, что участница из России полу...",Участница из России получает этот титул впервые.,Россиянка получает этот титул впервые.
2,"Уточняется, что финал прошел в Софии 4 февраля.",Финал прошел в Болгарии в начале февраля.,Финал был в Болгарии в феврале.,Финал был начале февраля. Он был в в Болгарии.
3,Участие в нем принимали 120 женщин из разных с...,В нем участвовали 120 женщин из разных стран.,В нем участвовали 120 женщин из разных стран.,В нем участвовали женщины из разных стран.
4,«Конкуренция на конкурсе была очень жесткая: р...,«В конкурсе было сложно выиграть. Было много у...,«В конкурсе было сложно выиграть. Было много у...,«В конкурсе было сложно выиграть.


In [18]:
new_data = pd.DataFrame(data["source"].head(200))

In [19]:
def apply_generation(text, level) -> str:
    prompt = format_prompt(user_text=text, level=level)
    return model.invoke(str(prompt), stop=["\n\n", " \n\n", ". \n\n"])

In [20]:
for level in [1, 2, 3]:
    column = f"level {level}"
    new_data[column] = new_data["source"].apply(lambda x: apply_generation(x, level))

In [21]:
new_data_location = base_location / "data" / "RuSimpleSentAphasia_200_generated_vikhr_few_shot.csv"

new_data.to_csv(new_data_location.as_posix(), index=False)

## Let's calculate BERTscore between ground truth and predicted texts

In [14]:
from evaluate import load
import numpy as np

bertscore = load("bertscore")

bert_scores = {}
val_data = data.head(200)


for level in val_data.columns[1:]:
    references = val_data[level].tolist()
    predictions = new_data[level].tolist()

    results = bertscore.compute(predictions=predictions, references=references, lang="ru")
    
    bert_scores[level] = {
        "precision": np.mean(results['precision']),
        "recall": np.mean(results['recall']),
        "f1": np.mean(results['f1'])
    }

In [15]:
for level, scores in bert_scores.items():
    print(f"BERTScore for {level}:")
    print(f"Precision: {scores['precision']}")
    print(f"Recall: {scores['recall']}")
    print(f"F1: {scores['f1']}\n")

BERTScore for level 1:
Precision: 0.7397541463375091
Recall: 0.7841574850678444
F1: 0.7605583503842354

BERTScore for level 2:
Precision: 0.7204789718985558
Recall: 0.7678983905911445
F1: 0.7425330486893654

BERTScore for level 3:
Precision: 0.6981816080212593
Recall: 0.7512632647156715
F1: 0.7229271700978279

